## Main Data loading pipeline

This NB is largely inspired by: https://github.com/rasbt/LLMs-from-scratch/blob/main/ch02/01_main-chapter-code/dataloader.ipynb

> **Origin and license.** This notebook is derived from [`ch02/01_main-chapter-code/dataloader.ipynb`](https://github.com/rasbt/LLMs-from-scratch/blob/main/ch02/01_main-chapter-code/dataloader.ipynb) in the repository [LLMs-from-scratch](https://github.com/rasbt/LLMs-from-scratch), Copyright 2023-2026 Sebastian Raschka, licensed under the [Apache License 2.0](../LICENSES/Apache-2.0-LLMs-from-scratch.txt).
>
> **Modified by L. Saetta (2026):** explanations rewritten and extended, and code adapted to this repository. See [NOTICE](../NOTICE).

In [1]:
import tiktoken
import torch
from torch.utils.data import Dataset, DataLoader

In [2]:
# the file with the text used to build the dataset
INPUT_FILE = "./the-verdict.txt"
# this is the value we will be using with tiktoken
ENCODING = "gpt2"

BATCH_SIZE = 8
MAX_LENGTH = 4
EMBEDDING_DIM = 256
CONTEXT_LENGTH = 1024

### Window size, stride, and context length

- **`max_length`** is the number of tokens in each dataset input sequence.
  The target sequence has the same length and is shifted forward by one token:
  for example, input `[t0, t1, t2, t3]` has target `[t1, t2, t3, t4]`.
  Creating one sample therefore requires at least `max_length + 1` tokens.
- **`stride`** is the number of tokens by which the window's starting position
  advances between dataset samples, before the dataloader optionally shuffles
  them. With `max_length = 4`, a stride of `2` gives input windows starting at
  positions `0, 2, 4, ...`, so adjacent windows overlap by two tokens.
  A stride of `4` gives non-overlapping input windows; a stride greater than `4`
  leaves gaps between input windows. Both values must be positive integers.
- **`context_length`** is the maximum sequence length supported by the positional
  embedding table in this notebook. It determines the number of learned position
  vectors, with valid position IDs from `0` to `context_length - 1`.
  In a full GPT model, the context length is the maximum number of tokens the
  model can process in one sequence. Here, only the embedding layers are built.

**The key distinction:** `max_length` selects the sequence length used by the
dataset, while `context_length` sets the positional embedding capacity.
For these sequences to fit, `max_length <= context_length`.
In this notebook, each input has **4 tokens**, and the position table can support
sequences of up to **1,024 tokens**. Only its first four position vectors are
used for each input. Setting `context_length = 1024` does not make the dataloader
produce 1,024-token sequences. The extra token needed to construct the targets
does not require an extra input position embedding.


In [3]:
# this is the Dataset that will encapsulate our text data and return
# in the format used for training
class GPTDatasetV1(Dataset):
    def __init__(self, text, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i : i + max_length]
            target_chunk = token_ids[i + 1 : i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk, dtype=torch.long))
            self.target_ids.append(torch.tensor(target_chunk, dtype=torch.long))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [4]:
def create_dataloader_v1(
    tokenizer,
    text,
    batch_size,
    max_length,
    stride,
    shuffle=True,
    drop_last=True,
    num_workers=0,
):

    # Create dataset
    dataset = GPTDatasetV1(text, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers,
    )

    return dataloader

In [5]:
with open(INPUT_FILE, "r", encoding="utf-8") as text_file:
    raw_text = text_file.read()

tokenizer = tiktoken.get_encoding(ENCODING)

dataloader = create_dataloader_v1(
    tokenizer, raw_text, batch_size=BATCH_SIZE, max_length=MAX_LENGTH, stride=MAX_LENGTH
)

vocab_size = tokenizer.max_token_value + 1
embedding_dim = EMBEDDING_DIM
context_length = CONTEXT_LENGTH

# Embeddings are initialized randomly and will be learned during training.
token_embedding_layer = torch.nn.Embedding(vocab_size, embedding_dim)
position_embedding_layer = torch.nn.Embedding(context_length, embedding_dim)

In [6]:
#
# let's have a look
#
for batch in dataloader:
    input_ids, target_ids = batch

    token_embeddings = token_embedding_layer(input_ids)
    position_embeddings = position_embedding_layer(torch.arange(MAX_LENGTH))

    input_embeddings = token_embeddings + position_embeddings

    break

print(input_embeddings.shape)

torch.Size([8, 4, 256])
